In [0]:
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DateType,TimestampType, FloatType

from pyspark.sql.functions import to_timestamp, date_format, col

from pyspark.sql.functions import *

from datetime import datetime

catalog_name = "openaq"

schema = "silver"

In [0]:
df_latest=spark.read.table(f"{catalog_name}.{schema}.slv_latest")
df_sensors=spark.read.table(f"{catalog_name}.{schema}.slv_sensors")

### Merging Latest and Sensors tables

In [0]:
df_sensors.columns

In [0]:
df_latest.columns

In [0]:
df_sensor_latest  = spark.sql(f""" 
SELECT 
    s.sensor_id, s.sensor_name, s.coverage_expected_count, s.coverage_observed_count, s.latest_latitude, s.latest_longitude, s.latest_value, s.parameter_id, s.parameter, s.parameter_display_name, s.parameter_units, s.summary_avg, s.summary_max, s.summary_median, s.summary_min, s.std_coverage_from_local, s.std_coverage_from_utc, s.std_coverage_to_local, s.std_coverage_to_utc, s.std_datetime_first_local, s.std_datetime_first_utc, s.std_datetime_last_local, s.std_datetime_last_utc, s.std_latest_datetime_local, s.std_latest_datetime_utc, s.std_coverage_expected_interval, s.std_coverage_observed_interval, s.std_coverage_percent_complete, s.std_coverage_percent_coverage, s.std_parameter_name,
    l.value, l.location_id, l.std_datetime_utc, l.std_datetime_local
FROM {catalog_name}.{schema}.slv_sensors AS s
INNER JOIN {catalog_name}.{schema}.slv_latest AS l
    ON s.sensor_id = l.sensor_id
""")

In [0]:
display(df_sensor_latest)

In [0]:
df_sensor_latest.createOrReplaceTempView("df_sensor_latest")
gold_latest_sensor_data  = spark.sql(f"""
SELECT 
    sl.*,
    gdc.*
FROM df_sensor_latest AS sl
LEFT JOIN {catalog_name}.gold.gld_dim_consolidated AS gdc
    ON sl.location_id = gdc.location_id
""")

### Merging Dim and Fact consolidated tables

In [0]:
df_sensor_latest.createOrReplaceTempView("df_sensor_latest")
gold_latest_sensor_data  = spark.sql(f"""
SELECT 
    sl.sensor_id, sl.sensor_name, sl.coverage_expected_count, sl.coverage_observed_count, sl.latest_latitude, sl.latest_longitude, sl.latest_value, sl.parameter_id, sl.parameter, sl.parameter_display_name, sl.parameter_units, sl.summary_avg, sl.summary_max, sl.summary_median, sl.summary_min, sl.std_coverage_from_local, sl.std_coverage_from_utc, sl.std_coverage_to_local, sl.std_coverage_to_utc, sl.std_datetime_first_local, sl.std_datetime_first_utc, sl.std_datetime_last_local, sl.std_datetime_last_utc, sl.std_latest_datetime_local, sl.std_latest_datetime_utc, sl.std_coverage_expected_interval, sl.std_coverage_observed_interval, sl.std_coverage_percent_complete, sl.std_coverage_percent_coverage, sl.std_parameter_name, sl.value, sl.location_id, sl.std_datetime_utc, sl.std_datetime_local,
    
    gdc.std_location_name, gdc.std_locality, gdc.country_id, gdc.std_country_name, gdc.std_country_code, gdc.owner_name, gdc.provider_id, gdc.isMobile, gdc.isMonitor, gdc.instrument_id, gdc.std_owner_name, gdc.std_provider_name, gdc.parameter_name, gdc.parameter_displayName, gdc.license_id, gdc.std_license_name, gdc.std_license_attribution_name, gdc.std_instrument_name, gdc.std_timezone, gdc.std_first_seen_utc, gdc.std_last_seen_utc, gdc.attributionRequired, gdc.commercialUseAllowed, gdc.modificationAllowed, gdc.redistributionAllowed, gdc.shareAlikeRequired, gdc.manufacturer_id, gdc.std_manufacturer_name
FROM df_sensor_latest AS sl
LEFT JOIN {catalog_name}.gold.gld_dim_consolidated AS gdc
    ON sl.location_id = gdc.location_id
""")

In [0]:
display(gold_latest_sensor_data)

In [0]:
gold_latest_sensor_data.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{catalog_name}.gold.gld_latest_sensor_data")

In [0]:
display(gold_latest_sensor_data)

In [0]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")

output_path = f"s3://openaq-project-dm/transformed_data/gld_latest_sensor_data_{timestamp}/"

gold_latest_sensor_data.write \
    .mode("overwrite") \
    .format("parquet") \
    .save(output_path)